# Segmentation tracking

## Imports

In [1]:
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
from PIL import Image
from IPython.display import HTML, display
import torch
import torch.nn.functional as F
import torchvision.transforms as TVT
import torchvision.transforms.functional as TVTF
from torch import Tensor, nn
from tqdm.auto import tqdm

## Resources location

**Note**: update **DINOV3_DIR** and **WEIGHTS_PATH** below 

In [14]:
DINOV3_DIR = Path('') # for example: /path/to/EchoDino_
WEIGHTS_PATH = Path('') # for example: /path/to/model.pth

DATA_DIR = Path('./resources').resolve() / 'segmentation_tracking_input'
SEQUENCE_NAME = 'patient0001_2CH_half_sequence'

OUTPUT_DIR =  Path('./resources').resolve() / 'segmentation_tracking_output'
OUTPUT_PREFIX = 'my_echo'

In [15]:
if not DINOV3_DIR or str(DINOV3_DIR).strip() == ".":
    raise ValueError("Please specify the path for 'DINOV3_DIR'")

if not WEIGHTS_PATH or str(WEIGHTS_PATH).strip() == ".":
    raise ValueError("Please specify the path for 'WEIGHTS_PATH'")

## Hyperparameters

In [3]:
MODEL_NAME = 'dinov3_vitl16'
CHECKPOINT_FORMAT = 'teacher'
DEVICE = 'cuda'

SHORT_SIDE = 2048
MAX_CONTEXT_LENGTH = 3
NEIGHBORHOOD_SIZE = 30
NEIGHBORHOOD_SHAPE = 'circle'
TOPK = 5
TEMPERATURE = 0.2
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

FPS = 4
OVERLAY_ALPHA = 0.5
DISPLAY_MAX_MB = 100

## Helper functions

In [4]:
def frame_number(path):
    return int(path.stem.rsplit('_', 1)[-1])


def load_video_frames(file_paths):
    frames = []
    for path in file_paths:
        with Image.open(path) as image:
            frames.append(image.convert('RGB'))
    return frames


class ResizeToMultiple(nn.Module):
    def __init__(self, short_side, multiple):
        super().__init__()
        self.short_side = short_side
        self.multiple = multiple

    def _round_up(self, side):
        return math.ceil(side / self.multiple) * self.multiple

    def forward(self, image):
        width, height = TVTF.get_image_size(image)
        if width > height:
            new_height = self._round_up(self.short_side)
            new_width = self._round_up(width * new_height / height)
        else:
            new_width = self._round_up(self.short_side)
            new_height = self._round_up(height * new_width / width)
        return TVTF.resize(
            image, [new_height, new_width],
            interpolation=TVT.InterpolationMode.BICUBIC,
        )


def mask_to_rgb(mask, num_masks):
    if isinstance(mask, Tensor):
        mask = mask.cpu().numpy()
    labels = np.asarray(mask, dtype=np.int64)
    foreground_count = num_masks - 1
    if foreground_count <= 10:
        colors = plt.get_cmap('tab10')(labels - 1)[..., :3]
    elif foreground_count <= 20:
        colors = plt.get_cmap('tab20')(labels - 1)[..., :3]
    else:
        colors = plt.get_cmap('gist_rainbow')(
            (labels - 1) / max(foreground_count - 1, 1)
        )[..., :3]
    colors = (colors * 255).astype(np.uint8)
    colors[labels == 0] = 0
    return colors


def overlay_mask(image, mask_rgb, alpha):
    blended = (1 - alpha) * image.astype(np.float32) + alpha * mask_rgb.astype(np.float32)
    return blended.clip(0, 255).astype(np.uint8)

In [5]:
@torch.inference_mode()
def forward(
    model: nn.Module,
    img: Tensor,  # [3, H, W] already normalized for the model
) -> Tensor:
    feats = model.get_intermediate_layers(img.unsqueeze(0), n=1, reshape=True)[0]  # [1, D, h, w]
    feats = feats.movedim(-3, -1)  # [1, h, w, D]
    feats = F.normalize(feats, dim=-1, p=2)
    return feats.squeeze(0)  # [h, w, D]

@torch.inference_mode()
def propagate(
    current_features: Tensor,  # [h", w", D], where h=h", w=w", and " stands for current
    context_features: Tensor,  # [t, h, w, D]
    context_probs: Tensor,  # [t, h, w, M]
    neighborhood_mask: Tensor,  # [h", w", h, w]
    topk: int,
    temperature: float,
) -> Tensor:
    t, h, w, M = context_probs.shape

    # Compute similarity current -> context
    dot = torch.einsum(
        "ijd, tuvd -> ijtuv",
        current_features,  # [h", w", D]
        context_features,  # [t, h, w, D]
    )  # [h", w", t, h, w]

    # Restrict focus to local neighborhood
    dot = torch.where(
        neighborhood_mask[:, :, None, :, :],  # [h", w", 1, h, w]
        dot,  # [h", w", t, h, w]
        -torch.inf,
    )

    # Select top-k patches inside the neighborhood
    dot = dot.flatten(2, -1).flatten(0, 1)  # [h"w", thw]
    k_th_largest = torch.topk(dot, dim=1, k=min(topk, dot.shape[1])).values  # [h"w", k]
    dot = torch.where(
        dot >= k_th_largest[:, -1:],  # [h"w", thw]
        dot,  # [h"w", thw]
        -torch.inf,
    )

    # Propagate probabilities from context to current frame
    weights = F.softmax(dot / temperature, dim=1)  # [h"w", thw]
    current_probs = torch.mm(
        weights,  # [h"w", thw]
        context_probs.flatten(0, 2),  # [thw, M]
    )  # [h"w", M]

    # Propagated probs should already sum to 1, but just in case
    current_probs = current_probs / current_probs.sum(dim=1, keepdim=True)  # [h"w", M]

    return current_probs.unflatten(0, (h, w))  # [h", w", M]


def make_neighborhood_mask(h: int, w: int, size: float, shape: str, device) -> Tensor:
    ij = torch.stack(
        torch.meshgrid(
            torch.arange(h, dtype=torch.float32, device=device),
            torch.arange(w, dtype=torch.float32, device=device),
            indexing="ij",
        ),
        dim=-1,
    )  # [h, w, 2]
    if shape == "circle":
        ord = 2
    elif shape == "square":
        ord = torch.inf
    else:
        raise ValueError(f"Invalid {shape=}")
    norm = torch.linalg.vector_norm(
        ij[:, :, None, None, :] - ij[None, None, :, :, :],  # [h", w", h, w, 2]
        ord=ord,
        dim=-1,
    )  # [h", w", h, w]
    mask = norm <= size  # [h", w", h, w] bool, True inside, False outside
    return mask

def postprocess_probs(
    probs: Tensor,  # [B, M, H', W']
) -> Tensor:
    vmin = probs.flatten(2, 3).min(dim=2).values  # [B, M]
    vmax = probs.flatten(2, 3).max(dim=2).values  # [B, M]
    probs = (probs - vmin[:, :, None, None]) / (vmax[:, :, None, None] - vmin[:, :, None, None])
    probs = torch.nan_to_num(probs, nan=0)
    return probs  # [B, M, H', W']

In [6]:
def show_video(video, fps, title):
    fig, ax = plt.subplots()
    image = ax.imshow(video[0])
    ax.axis('off')
    ax.set_title(title)

    def update(index):
        image.set_data(video[index])
        return (image,)

    clip = animation.FuncAnimation(
        fig, update, frames=len(video), interval=1000 / fps, blit=True,
    )
    plt.close(fig)
    return HTML(clip.to_jshtml())


def save_mp4(video, path, fps):
    video = np.asarray(video)
    if video.ndim != 4 or video.shape[-1] != 3 or video.dtype != np.uint8:
        raise ValueError('Video must be uint8 RGB with shape (T, H, W, 3).')
    _, height, width, _ = video.shape
    # MP4 encoders require even dimensions.
    output_width = width + width % 2
    output_height = height + height % 2
    writer = cv2.VideoWriter(
        str(path), cv2.VideoWriter_fourcc(*'mp4v'), fps,
        (output_width, output_height),
    )
    if not writer.isOpened():
        writer.release()
        raise RuntimeError(f'Cannot open video writer: {path}')
    try:
        for frame in video:
            if output_width != width or output_height != height:
                frame = np.pad(
                    frame, ((0, height % 2), (0, width % 2), (0, 0)), mode='edge',
                )
            writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    finally:
        writer.release()
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f'Video was not saved: {path}')

## Model loading

In [7]:
if not (DINOV3_DIR / 'hubconf.py').is_file():
    raise FileNotFoundError(DINOV3_DIR / 'hubconf.py')
if not WEIGHTS_PATH.is_file():
    raise FileNotFoundError(WEIGHTS_PATH)
if CHECKPOINT_FORMAT not in {'teacher', 'backbone'}:
    raise ValueError('CHECKPOINT_FORMAT must be teacher or backbone.')

device = torch.device(DEVICE)
if device.type == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable.')

model = torch.hub.load(
    str(DINOV3_DIR), MODEL_NAME, source='local', pretrained=False,
)
checkpoint = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=True)
if CHECKPOINT_FORMAT == 'teacher':
    state_dict = {
        key.removeprefix('backbone.'): value
        for key, value in checkpoint['teacher'].items()
        if 'head' not in key
    }
else:
    state_dict = checkpoint
model.load_state_dict(state_dict, strict=True)
del checkpoint, state_dict
model = model.to(device).eval()
patch_size = model.patch_size
print(f'Model: {MODEL_NAME} | Device: {device}')

Model: dinov3_vitl16 | Device: cuda


## Data loading

In [8]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(DATA_DIR)
if SHORT_SIDE <= 0 or MAX_CONTEXT_LENGTH < 0 or TOPK <= 0 or TEMPERATURE <= 0:
    raise ValueError('Invalid tracking hyperparameters.')
if NEIGHBORHOOD_SIZE < 0 or NEIGHBORHOOD_SHAPE not in {'circle', 'square'}:
    raise ValueError('Invalid neighborhood settings.')
if FPS <= 0 or not 0 <= OVERLAY_ALPHA <= 1:
    raise ValueError('Invalid video settings.')

sequence_paths = list(DATA_DIR.glob(f'{SEQUENCE_NAME}_*.png'))
image_paths = sorted(
    (path for path in sequence_paths if 'gt' not in path.stem),
    key=frame_number,
)
gt_paths = sorted(
    (path for path in sequence_paths if 'gt' in path.stem),
    key=frame_number,
)
if not image_paths or not gt_paths:
    raise FileNotFoundError(f'Frames or masks missing: {DATA_DIR / SEQUENCE_NAME}')
if frame_number(image_paths[0]) != frame_number(gt_paths[0]):
    raise ValueError('The first frame and first mask must have matching indices.')

frames = load_video_frames(image_paths)
num_frames = len(frames)
original_width, original_height = frames[0].size
if any(frame.size != frames[0].size for frame in frames):
    raise ValueError('All frames must have the same dimensions.')
with Image.open(gt_paths[0]) as image:
    first_mask_np = np.array(image)
if first_mask_np.ndim != 2 or not np.issubdtype(first_mask_np.dtype, np.integer):
    raise ValueError('The first mask must contain 2D integer labels.')
if first_mask_np.min() < 0 or first_mask_np.max() > 255:
    raise ValueError('Mask labels must be in [0, 255].')
mask_height, mask_width = first_mask_np.shape
if (mask_width, mask_height) != (original_width, original_height):
    raise ValueError('Frame and mask dimensions must match.')
num_masks = int(first_mask_np.max()) + 1

transform = TVT.Compose([
    ResizeToMultiple(SHORT_SIDE, patch_size),
    TVT.ToTensor(),
    TVT.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])
print(f'Frames: {num_frames} | Size: {original_width} x {original_height} | Labels: {num_masks}')

Frames: 18 | Size: 549 x 389 | Labels: 4


## Segmentation tracking

In [9]:
with torch.inference_mode():
    first_frame = transform(frames[0]).to(device)
    first_feats = forward(model, first_frame)
    feats_height, feats_width = first_feats.shape[:2]

    first_mask = torch.from_numpy(first_mask_np.astype(np.int64)).to(device)
    first_mask = F.interpolate(
        first_mask[None, None].float(),
        size=(feats_height, feats_width), mode='nearest-exact',
    )[0, 0].long()
    first_probs = F.one_hot(first_mask, num_masks).float()
    neighborhood_mask = make_neighborhood_mask(
        feats_height, feats_width, NEIGHBORHOOD_SIZE,
        NEIGHBORHOOD_SHAPE, device,
    )

    mask_predictions = np.zeros(
        (num_frames, mask_height, mask_width), dtype=np.uint8,
    )
    mask_predictions[0] = first_mask_np
    features_queue = []
    probs_queue = []

    for frame_idx in tqdm(range(1, num_frames), desc='Tracking'):
        current_frame = transform(frames[frame_idx]).to(device)
        current_feats = forward(model, current_frame)
        context_feats = torch.stack([first_feats, *features_queue])
        context_probs = torch.stack([first_probs, *probs_queue])
        current_probs = propagate(
            current_feats, context_feats, context_probs,
            neighborhood_mask, TOPK, TEMPERATURE,
        )

        features_queue.append(current_feats)
        probs_queue.append(current_probs)
        if len(features_queue) > MAX_CONTEXT_LENGTH:
            features_queue.pop(0)
            probs_queue.pop(0)

        upsampled_probs = F.interpolate(
            current_probs.movedim(-1, -3).unsqueeze(0),
            size=(mask_height, mask_width), mode='nearest',
        )
        upsampled_probs = postprocess_probs(upsampled_probs).squeeze(0)
        mask_predictions[frame_idx] = (
            upsampled_probs.argmax(0).to(torch.uint8).cpu().numpy()
        )

Tracking:   0%|          | 0/17 [00:00<?, ?it/s]

In [10]:
input_video = np.stack([np.array(frame) for frame in frames])
pred_video = mask_to_rgb(mask_predictions, num_masks)
overlay_video = np.stack([
    overlay_mask(frame, mask, OVERLAY_ALPHA)
    for frame, mask in zip(input_video, pred_video)
])

plt.rcParams['animation.embed_limit'] = DISPLAY_MAX_MB
display(show_video(overlay_video, FPS, 'Overlay'))
display(show_video(input_video, FPS, 'Input'))

## Saving

In [11]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_paths = {}
for name, video in (
    ('input', input_video), ('overlay', overlay_video),
):
    path = OUTPUT_DIR / f'{name}_{OUTPUT_PREFIX}.mp4'
    save_mp4(video, path, FPS)
    output_paths[name] = path
    print(path.resolve())

/data/sc159/EchoDino/analysis/videoTrack/temp/resources/segmentation_tracking_output/input_my_echo.mp4
/data/sc159/EchoDino/analysis/videoTrack/temp/resources/segmentation_tracking_output/overlay_my_echo.mp4
